# Varying light counts ablation


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.06
n_iter = 250

In [ ]:
global_seed = 2 # Can be None

In [ ]:
import open_clip
from losses.clip import CLIPDirectionalCosineSimilarity
from utils.train import train_with_criterion
from torchvision.transforms.v2 import RandomChoice, RandomPerspective, RandomResizedCrop, RandomHorizontalFlip, GaussianBlur, Identity, Transform
from examples.example_scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, CandleScene, CarStudioScene
import os

clip_model_name = 'ViT-B-16-SigLIP-512'
clip_pretrained = 'webli'
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

def learning_rate_scheduler_creator(optimizer: torch.optim.Optimizer) -> torch.optim.lr_scheduler.LRScheduler:
    n_warmup_steps = 50
    warmup = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: min((step + 1) / n_warmup_steps, 1.0))
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_iter - n_warmup_steps, eta_min=1e-6)
    return torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[n_warmup_steps])

from utils.model.model_utils import create_clip_model_and_tokenizer


scenes_to_test = [
    CarStudioScene(configuration='dome_lights', device=device),
    CarStudioScene(configuration='four_small_area_lights', device=device),
    CarStudioScene(configuration='single_sun_light', device=device),
]


loss_type ='clip_directional_cosine_similarity'

def get_fine_tuned_name_from_path(checkpoint_path: str, make_human_readable: bool = False) -> str:
    base_name = os.path.basename(os.path.dirname(checkpoint_path))
    if make_human_readable:
        return base_name.replace('_', ' ').title()
    return base_name

fine_tune_name = "siglip_blend-training-data_64-output-dim.pt"
model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
    clip_model_name,
    device=device,
    pretrained=clip_pretrained,
    fine_tune=fine_tune_name
)

model.eval()
print("Loaded fine-tuned model from:", fine_tune_name)

output_directory = "varying_light_counts_ablation"
for scene in scenes_to_test:
    if loss_type == 'clip_directional_cosine_similarity':
        initial_prompt = 'ugly, uninteresting lighting'
        target_prompt = 'a bright sunny day lighting'
        criterion = CLIPDirectionalCosineSimilarity(initial_prompt, target_prompt, scene.get_combined_image(color_space_converter).permute(2, 1, 0), model, tokenizer, device=device, preprocess=preprocess_eval, always_prenormalize_vectors=True)
        try:
            title_prefix:str = "ImageTextFineTuning '" + get_fine_tuned_name_from_path(fine_tune_name, make_human_readable=True) + "'"
        except Exception as e:
            title_prefix:str = "ImageTextFineTuning"
    else:
        raise ValueError("Unsupported loss type: " + loss_type)

    size = model.visual.preprocess_cfg['size'] or (224, 224)

    train_with_criterion(
        scene,
        lr, n_iter, criterion,
        starting_multiplier_std=(0.1, 0.1, 0.1),
        output_subdirectory_name=output_directory,
        n_results=4,
        torch_precision=torch_precision,
        augmentation=RandomResizedCrop(size=size, scale=(0.3, 1.0), antialias=True),
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=title_prefix + " Fine-Tuned Model",
        device=device,
        save_every=50,
        model_name=get_fine_tuned_name_from_path(fine_tune_name),
        pretrained_source=fine_tune_name,
        seed=global_seed,
        show_images_after_augmentation=True
    )

## Result Collection Helper
Run the following cell to collect all loss plots and final image grids from a specific output directory into a single summary folder. This makes it easier to compare runs side-by-side.

In [ ]:
from utils.record_keeping.experiment import FolderManager

# Initialize FolderManager with the subdirectory and call collect_results
folder_manager = FolderManager(output_directory)
folder_manager.collect_results()